ChatGPI - API - LAB

In [ ]:
#pip install datasets

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from pathlib import Path
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
    
# Create images directory
 
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

/Users/juliaguth/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/juliaguth/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Loading product dataset...


✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [2]:
from pathlib import Path

images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)

print("Saving all images...")

for i, item in enumerate(dataset):
    try:
        image = item["image"]
        image_path = images_dir / f"product_{i}.jpg"
        image.save(image_path)
    except Exception as e:
        print(f"Error at image {i}: {e}")

print("Done saving.")

Saving all images...
Done saving.


In [3]:
import base64

def encode_image(image_path):
    """
    Encodes an image file to base64 string.
    
    Args:
        image_path (str or Path): Path to the image file
        
    Returns:
        str: Base64 encoded string
    """
    try:
        with open(image_path, "rb") as image_file:
            encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
        return encoded_string
    
    except FileNotFoundError:
        print(f"❌ File not found: {image_path}")
        return None
    
    except Exception as e:
        print(f"❌ Error encoding image: {e}")
        return None

In [4]:
import json

def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating structured product listings.

    Parameters:
    - product_name (str): Name of the product
    - price (float): Price of the product
    - category (str): Product category
    - additional_info (str, optional): Extra product details

    Returns:
    - str: Formatted prompt
    """

    # Clean inputs
    product_name = str(product_name).strip()
    category = str(category).strip()
    additional_info = str(additional_info).strip() if additional_info else None

    # Format optional section
    additional_section = f"- Additional Info: {additional_info}\n" if additional_info else ""

    prompt = f"""
You are an expert e-commerce copywriter.

Analyze the product image together with the product metadata and generate a compelling, accurate, and professional product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{additional_section}
Instructions:
Create a product listing with the following fields:

1. title
- catchy and SEO-friendly
- maximum 60 characters

2. description
- 150 to 200 words
- persuasive but realistic
- highlight visible features, benefits, colors, materials, design elements, and likely use cases
- do not invent technical specifications that are not visible or not provided

3. features
- 5 to 7 bullet-point style items
- short, clear, and benefit-oriented

4. keywords
- 10 to 15 SEO keywords
- relevant to the product and category

Important rules:
- Base your answer only on the image and provided metadata
- If something is unclear, stay general rather than making up details
- Return valid JSON only
- Do not include markdown
- Do not include explanations before or after the JSON

Use exactly this JSON structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2", "Feature 3"],
    "keywords": ["keyword1", "keyword2", "keyword3"]
}}
"""
    return prompt.strip()


# Test prompt creation
test_prompt = create_product_listing_prompt(
    product_name="Wireless Bluetooth Headphones",
    price=79.99,
    category="Electronics",
    additional_info="Noise cancelling, 30-hour battery"
)

print("\n" + "=" * 50)
print("PROMPT TEMPLATE")
print("=" * 50)
print(test_prompt[:1000] + "...")


PROMPT TEMPLATE
You are an expert e-commerce copywriter.

Analyze the product image together with the product metadata and generate a compelling, accurate, and professional product listing.

Product Information:
- Name: Wireless Bluetooth Headphones
- Price: $79.99
- Category: Electronics
- Additional Info: Noise cancelling, 30-hour battery

Instructions:
Create a product listing with the following fields:

1. title
- catchy and SEO-friendly
- maximum 60 characters

2. description
- 150 to 200 words
- persuasive but realistic
- highlight visible features, benefits, colors, materials, design elements, and likely use cases
- do not invent technical specifications that are not visible or not provided

3. features
- 5 to 7 bullet-point style items
- short, clear, and benefit-oriented

4. keywords
- 10 to 15 SEO keywords
- relevant to the product and category

Important rules:
- Base your answer only on the image and provided metadata
- If something is unclear, stay general rather than mak

In [ ]:
#pip install openai

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [8]:


import os
import json
import base64
from dotenv import load_dotenv
from pathlib import Path
from openai import OpenAI


# -----------------------------
# 1. OpenAI client
# -----------------------------
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


# -----------------------------
# 2. Encode image to base64
# -----------------------------
def encode_image(image_path):
    """
    Encode an image file to a base64 string.

    Parameters:
    - image_path (str or Path): path to image

    Returns:
    - str: base64-encoded image string
    """
    image_path = Path(image_path)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


# -----------------------------
# 3. Prompt creation
# -----------------------------
def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Build the text prompt for the model.
    """
    additional_section = f"- Additional Info: {additional_info}\n" if additional_info else ""

    prompt = f"""
You are an expert e-commerce copywriter.

Analyze the product image together with the product metadata and generate a compelling, accurate, and professional product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{additional_section}
Instructions:
Create a product listing with the following fields:

1. title
- catchy and SEO-friendly
- maximum 60 characters

2. description
- 150 to 200 words
- persuasive but realistic
- highlight visible features, benefits, colors, materials, design elements, and likely use cases
- do not invent technical specifications that are not visible or not provided

3. features
- 5 to 7 items
- short, clear, and benefit-oriented

4. keywords
- 10 to 15 SEO keywords
- relevant to the product and category

Important rules:
- Base your answer only on the image and provided metadata
- If something is unclear, stay general rather than making up details
- Return the result in the requested JSON structure only
""".strip()

    return prompt


# -----------------------------
# 4. JSON schema for strict output
# -----------------------------
PRODUCT_LISTING_SCHEMA = {
    "name": "product_listing",
    "strict": True,
    "schema": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string"
            },
            "description": {
                "type": "string"
            },
            "features": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 5,
                "maxItems": 7
            },
            "keywords": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 10,
                "maxItems": 15
            }
        },
        "required": ["title", "description", "features", "keywords"],
        "additionalProperties": False
    }
}


# -----------------------------
# 5. Main API call
# -----------------------------
def generate_product_listing(image_path, product_name, price, category, additional_info=None, model="gpt-4.1"):
    """
    Send image + prompt to OpenAI and return parsed JSON output.

    Parameters:
    - image_path (str or Path)
    - product_name (str)
    - price (float)
    - category (str)
    - additional_info (str, optional)
    - model (str): OpenAI model name

    Returns:
    - dict: parsed JSON product listing
    """
    prompt = create_product_listing_prompt(
        product_name=product_name,
        price=price,
        category=category,
        additional_info=additional_info
    )

    base64_image = encode_image(image_path)

    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": [
                    {"type": "input_text", "text": prompt},
                    {
                        "type": "input_image",
                        "image_url": f"data:image/jpeg;base64,{base64_image}",
                        "detail": "high"
                    }
                ]
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": PRODUCT_LISTING_SCHEMA["name"],
                "strict": PRODUCT_LISTING_SCHEMA["strict"],
                "schema": PRODUCT_LISTING_SCHEMA["schema"]
            }
        }
    )

    # The API returns text output; with structured output requested, this should be valid JSON
    raw_text = response.output_text
    parsed = json.loads(raw_text)

    return parsed


# -----------------------------
# 6. Test run
# -----------------------------
# -----------------------------
# 6. Test run
# -----------------------------
try:
    result = generate_product_listing(
        image_path="product_images/product_0.jpg",
        product_name="Fashion Product",
        price=79.99,
        category="Fashion",
        additional_info="Image from Hugging Face dataset"
    )

    print("✓ API call succeeded")
    print("✓ Response received")
    print("✓ JSON parsed correctly\n")

    print(json.dumps(result, indent=4))

except FileNotFoundError as e:
    print(f"Image error: {e}")

except json.JSONDecodeError as e:
    print(f"JSON parsing error: {e}")

except Exception as e:
    print(f"API/request error: {e}")

✓ API call succeeded
✓ Response received
✓ JSON parsed correctly

{
    "title": "Classic Checked Button-Down Shirt for Men",
    "description": "Elevate your everyday style with this classic checked button-down shirt, designed for the modern man seeking versatility and comfort. Featuring a timeless checkered pattern in blue and white hues, this shirt easily pairs with jeans or chinos for a smart-casual look perfect for work, outings, or weekend gatherings. The collared neckline and half sleeves provide a relaxed yet polished appearance, while the shirt\u2019s tailored fit ensures you look sharp without sacrificing comfort. Crafted from a breathable fabric, this piece offers day-long wearability and effortless style. Whether you're heading to the office, meeting friends, or enjoying a casual evening out, this button-down shirt is a wardrobe essential that transitions seamlessly through various occasions.",
    "features": [
        "Classic checkered pattern",
        "Comfortable half

PArt 6

In [10]:
import time
import pandas as pd
import json
from pathlib import Path


def process_multiple_products(dataset, max_products=5, output_file="generated_product_listings.json"):
    """
    Generate product listings for multiple products and save results.
    """

    results = []
    errors = []

    for i in range(max_products):
        print(f"\nProcessing product {i + 1}/{max_products}...")

        try:
            product = dataset[i]

            image_path = f"product_images/product_{i}.jpg"
            product_name = product.get("productDisplayName", f"Fashion Product {i}")
            category = product.get("articleType", "Fashion")
            price = 49.99

            additional_info = (
                f"Gender: {product.get('gender', 'Unknown')}, "
                f"Category: {product.get('masterCategory', 'Unknown')}, "
                f"Subcategory: {product.get('subCategory', 'Unknown')}, "
                f"Color: {product.get('baseColour', 'Unknown')}, "
                f"Season: {product.get('season', 'Unknown')}, "
                f"Usage: {product.get('usage', 'Unknown')}"
            )

            listing = generate_product_listing(
                image_path=image_path,
                product_name=product_name,
                price=price,
                category=category,
                additional_info=additional_info
            )

            result = {
                "product_id": product.get("id", i),
                "product_name": product_name,
                "price": price,
                "category": category,
                "image_path": image_path,
                "generated_listing": listing
            }

            results.append(result)
            print(f"✓ Success: {product_name}")

            # small pause to avoid sending requests too quickly
            time.sleep(1)

        except Exception as e:
            error_info = {
                "product_index": i,
                "error": str(e)
            }
            errors.append(error_info)
            print(f"⚠ Error processing product {i}: {e}")

    # Save successful results
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=4, ensure_ascii=False)

    # Save errors separately
    with open("product_listing_errors.json", "w", encoding="utf-8") as f:
        json.dump(errors, f, indent=4, ensure_ascii=False)

    print("\nBatch processing complete!")
    print(f"Successful listings: {len(results)}")
    print(f"Errors: {len(errors)}")
    print(f"Saved results to: {output_file}")
    print("Saved errors to: product_listing_errors.json")

    return results, errors

In [12]:
results, errors = process_multiple_products(
    dataset=dataset,
    max_products=5,
    output_file="generated_product_listings.json"
)
import json

with open("generated_product_listings.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=4, ensure_ascii=False)

print("✅ JSON file saved successfully!")


Processing product 1/5...
✓ Success: Turtle Check Men Navy Blue Shirt

Processing product 2/5...
✓ Success: Peter England Men Party Blue Jeans

Processing product 3/5...
✓ Success: Titan Women Silver Watch

Processing product 4/5...
✓ Success: Manchester United Men Solid Black Track Pants

Processing product 5/5...
✓ Success: Puma Men Grey T-shirt

Batch processing complete!
Successful listings: 5
Errors: 0
Saved results to: generated_product_listings.json
Saved errors to: product_listing_errors.json
✅ JSON file saved successfully!


Report: So basically, the way the API works in this project is pretty straightforward.
I take an image of a product and some basic information like the name, price, and category. Then I convert the image into a format the API understands (base64), and send both the image and the text prompt to the OpenAI API.

The API then analyzes the image together with the prompt and generates a full product listing — including a title, description, features, and keywords. It sends the result back as a JSON response, which I then parse in Python and store for later use.

I definitely ran into quite a few challenges along the way.

At the beginning, I had trouble loading and handling the dataset properly. The images were not automatically saved, so I had to figure out how to extract them from the Hugging Face dataset and store them locally.

Another big issue was with the API setup. I had created an .env file, but I named it incorrectly, so the API key wasn’t being recognized. That took a bit of time to debug, but once I fixed the environment variable loading, the API started working.

Towards the end, I also had difficulties generating and saving the JSON files correctly. Sometimes variables were not defined, or the file wasn’t being created properly, so I had to re-run parts of the pipeline and make sure everything was executed in the right order.

Once everything was working, the quality of the generated product listings was actually good.